# Reinforcement Learning for Beginners

### What if a computer could learn to play games... just like you do?

When you were little, you learned that touching a hot stove = **OUCH** (bad!) and eating ice cream = **YUM** (good!).  
You learned from **rewards** and **punishments** -- and that's exactly how **Reinforcement Learning (RL)** works!

---

## The 4 Big Ideas of RL

| Concept | Real Life Example | In RL |
|---------|-------------------|-------|
| **Agent** | You! | The computer/robot that learns |
| **Environment** | The world around you | The game or problem |
| **Action** | Turn left, turn right, jump | Choices the agent can make |
| **Reward** | Getting a gold star (+1) or losing a life (-1) | A score that tells the agent how well it did |

---
## Game 1: Train a Robot Dog!

Imagine you have a robot dog. It doesn't know any tricks yet.  
When it does something good, you give it a treat (+1).  
When it does something bad, you say "no" (-1).  

Over time, the robot dog **learns which actions get treats!**

Let's simulate this:

In [ ]:
import random

# Our robot dog can do 3 things
actions = ["Sit", "Bark", "Roll Over"]

# The owner wants the dog to "Sit"
correct_action = "Sit"

# The dog's brain: how good does it think each action is?
# It starts knowing nothing (all zeros)
brain = {"Sit": 0, "Bark": 0, "Roll Over": 0}

print("Robot Dog's brain BEFORE training:")
print(brain)
print()

In [ ]:
# Let's train the dog for 100 rounds!
total_treats = 0

for round_num in range(1, 101):
    # The dog picks an action
    # Early on: random guesses. Later: picks what it thinks is best.
    if random.random() < 0.3:  # 30% of the time, try something random (explore!)
        choice = random.choice(actions)
    else:  # 70% of the time, pick the best known action (exploit!)
        choice = max(brain, key=brain.get)

    # Did the dog do the right thing?
    if choice == correct_action:
        reward = +1  # Treat!
        total_treats += 1
    else:
        reward = -1  # No treat

    # Update the dog's brain -- learn from the reward
    brain[choice] = brain[choice] + 0.1 * reward

    # Print progress every 20 rounds
    if round_num % 20 == 0:
        print(f"Round {round_num:3d} | Choice: {choice:10s} | Treats so far: {total_treats}")

print()
print("Robot Dog's brain AFTER training:")
for action, score in brain.items():
    bar = '+' * int(max(score, 0) * 10)
    print(f"  {action:10s} -> score: {score:+.1f}  {bar}")

### What just happened?

- The dog started **clueless** (all scores = 0)
- It tried random things and learned from rewards
- After training, **"Sit" has the highest score** because it got the most treats!

> **Key Idea:** The agent (dog) learned the best action by **trial and error** -- not because we programmed the answer!

---
## Explore vs Exploit: The Ice Cream Problem

Imagine you're at an ice cream shop with 5 flavors. You've tried chocolate and loved it.

Do you:
- **Exploit**: Pick chocolate again (safe, you know it's good!)
- **Explore**: Try a new flavor (risky, but maybe you'll find something even better!)

RL agents face this same dilemma every single time they make a decision!

Let's see what happens with different explore vs exploit strategies:

In [ ]:
# Secret tastiness scores (the agent doesn't know these!)
flavors = {
    "Chocolate":    7,
    "Vanilla":      6,
    "Strawberry":   8,   # This is actually the BEST!
    "Mint":         5,
    "Cookie Dough": 9,   # Secret champion!
}

def ice_cream_experiment(explore_rate, num_days=50):
    """Simulate picking ice cream over many days."""
    beliefs = {f: 0 for f in flavors}  # Agent's guess of how good each flavor is
    total_happiness = 0

    for day in range(num_days):
        # Explore or exploit?
        if random.random() < explore_rate:
            pick = random.choice(list(flavors.keys()))  # Try random flavor
        else:
            pick = max(beliefs, key=beliefs.get)  # Pick current favorite

        # Get happiness (with some randomness, like real life!)
        happiness = flavors[pick] + random.randint(-2, 2)
        total_happiness += happiness

        # Learn
        beliefs[pick] = beliefs[pick] + 0.3 * (happiness - beliefs[pick])

    return total_happiness, max(beliefs, key=beliefs.get)


print("What happens with different explore rates?\n")
print(f"{'Strategy':<25} {'Happiness':>10} {'Favorite Found':>16}")
print("-" * 55)

random.seed(42)
for rate, name in [(0.0, "Never explore (0%)"),
                   (0.1, "Rarely explore (10%)"),
                   (0.3, "Sometimes explore (30%)"),
                   (1.0, "Always explore (100%)")]:
    happiness, fav = ice_cream_experiment(rate)
    print(f"  {name:<23} {happiness:>8}    {fav:>14}")

### What did we learn?

- **Never exploring (0%)** = You stick with the first thing that seems OK. You might miss the best option!
- **Always exploring (100%)** = You keep trying random things and never use what you learned.
- **The sweet spot** is somewhere in the middle -- explore enough to find good options, then exploit them!

> **Key Idea:** Good RL agents balance **exploration** (trying new things) and **exploitation** (using what works).

---
## Game 2: Treasure Hunt on a Grid!

Now let's build something more visual. Our agent is on a small grid and needs to find treasure while avoiding traps.

```
 _______________
|   |   |   |   |
| A |   |   | X |    A = Agent (start)
|___|___|___|___|
|   |   |   |   |    X = Trap  (-10 points)
|   |   | X |   |
|___|___|___|___|    
|   |   |   |   |
|   |   |   | T |    T = Treasure (+10 points)
|___|___|___|___|
```

The agent can move: **Up, Down, Left, Right**

Let's watch it learn!

In [ ]:
import numpy as np

# Grid setup
ROWS, COLS = 3, 4
START = (0, 0)
TREASURE = (2, 3)
TRAPS = [(0, 3), (1, 2)]

ACTIONS = ["Up", "Down", "Left", "Right"]
MOVES = {"Up": (-1, 0), "Down": (1, 0), "Left": (0, -1), "Right": (0, 1)}


def get_reward(position):
    if position == TREASURE:
        return 10
    elif position in TRAPS:
        return -10
    else:
        return -1  # Small penalty for each step (encourages finding treasure fast)


def move(position, action):
    dr, dc = MOVES[action]
    new_r = max(0, min(ROWS - 1, position[0] + dr))
    new_c = max(0, min(COLS - 1, position[1] + dc))
    return (new_r, new_c)


# Q-Table: the agent's brain!
# For every position + action, store how good that combo is
q_table = np.zeros((ROWS, COLS, len(ACTIONS)))

print("The agent's brain is empty -- all zeros!")
print(f"Brain shape: {q_table.shape}  (3 rows x 4 cols x 4 actions)")
print("\nLet's train it...")

In [ ]:
# Training!
learning_rate = 0.5
discount = 0.9       # How much the agent cares about future rewards
explore_rate = 0.3
episodes = 500

rewards_per_episode = []

for episode in range(episodes):
    pos = START
    total_reward = 0

    for step in range(20):  # Max 20 steps per episode
        # Pick action
        if random.random() < explore_rate:
            action_idx = random.randint(0, 3)
        else:
            action_idx = int(np.argmax(q_table[pos[0], pos[1]]))

        action = ACTIONS[action_idx]
        new_pos = move(pos, action)
        reward = get_reward(new_pos)
        total_reward += reward

        # Q-Learning update (the magic formula!)
        old_value = q_table[pos[0], pos[1], action_idx]
        future_best = np.max(q_table[new_pos[0], new_pos[1]])
        q_table[pos[0], pos[1], action_idx] = old_value + learning_rate * (
            reward + discount * future_best - old_value
        )

        pos = new_pos

        # End episode if we found treasure or hit a trap
        if pos == TREASURE or pos in TRAPS:
            break

    rewards_per_episode.append(total_reward)

print(f"Training complete! ({episodes} episodes)")
print(f"Average reward (first 50 episodes):  {np.mean(rewards_per_episode[:50]):.1f}")
print(f"Average reward (last 50 episodes):   {np.mean(rewards_per_episode[-50:]):.1f}")
print("\nThe agent got MUCH better over time!")

In [ ]:
# Let's visualize what the agent learned!

arrow = {"Up": "^  ", "Down": "v  ", "Left": "<  ", "Right": ">  "}

print("What the agent learned (best action at each position):")
print("=" * 30)

for r in range(ROWS):
    row_str = "| "
    for c in range(COLS):
        if (r, c) == TREASURE:
            row_str += " T  | "
        elif (r, c) in TRAPS:
            row_str += " X  | "
        else:
            best_action = ACTIONS[int(np.argmax(q_table[r, c]))]
            row_str += f" {arrow[best_action]} | "
    print(row_str)
    print("-" * 30)

print()
print("The arrows show which way the agent wants to go at each spot.")
print("Notice: it avoids traps (X) and heads toward treasure (T)!")

In [ ]:
# Watch the trained agent play!

print("Watching the trained agent find the treasure:\n")

pos = START
path = [pos]

for step in range(10):
    action_idx = int(np.argmax(q_table[pos[0], pos[1]]))
    action = ACTIONS[action_idx]
    pos = move(pos, action)
    path.append(pos)
    print(f"  Step {step+1}: Move {action:5s} -> Position {pos}", end="")

    if pos == TREASURE:
        print("  ** FOUND THE TREASURE! **")
        break
    elif pos in TRAPS:
        print("  ** OH NO, A TRAP! **")
        break
    else:
        print()

---
## Let's See the Learning Progress

In [ ]:
# Simple text-based chart of learning progress

print("Reward over time (each dot = average of 50 episodes):\n")

chunk_size = 50
averages = []
for i in range(0, len(rewards_per_episode), chunk_size):
    avg = np.mean(rewards_per_episode[i:i+chunk_size])
    averages.append(avg)

min_r = min(averages)
max_r = max(averages)

for i, avg in enumerate(averages):
    bar_len = int((avg - min_r) / (max_r - min_r + 0.01) * 40)
    bar = '#' * bar_len
    print(f"  Episodes {i*chunk_size+1:3d}-{(i+1)*chunk_size:3d} | {bar:<40s} | avg reward: {avg:.1f}")

print()
print("The bars get longer = the agent is getting smarter!")

---
## Quick Recap

| What We Learned | Example |
|---|---|
| RL agents learn by **trial and error** | Robot dog tried random tricks until it found what works |
| Agents need to **explore AND exploit** | Ice cream problem -- try new things but also use what you know |
| A **Q-Table** stores what the agent has learned | Grid game -- the agent remembered which moves are good at each spot |
| **Rewards shape behavior** | +10 for treasure, -10 for traps, -1 for each step |
| Agents get **better over time** | The reward chart shows improvement with practice |

---

## Where is RL Used in Real Life?

- **Video Games** -- AI that learns to play Atari, Chess, and Go
- **Self-Driving Cars** -- Learning to steer, brake, and accelerate
- **Robots** -- Learning to walk, pick up objects, and navigate
- **Recommendations** -- YouTube/Netflix suggesting what to watch next
- **ChatGPT & AI Assistants** -- RL helps make AI responses more helpful!

---

## Try It Yourself!

Go back and try changing:
1. The **explore rate** in the ice cream experiment -- what happens at 50%?
2. The **trap positions** in the grid game -- can the agent still find treasure?
3. The **reward values** -- what if traps give -100 instead of -10?

The best way to learn is to **experiment** -- just like an RL agent!